# 模型编译缓存功能

## 1. 功能简介

`torch.compile` 采用即时编译（Just-In-Time，JIT）机制，首次成图通常包含较明显的编译开销。在线推理、弹性扩容和故障拉起等场景对冷启动时延敏感，因此可以使用编译缓存缩短服务启动后的首次推理时间。

从流程上看，首次成图耗时主要包括两部分：Dynamo 捕获并生成 FX 图，以及后端基于 FX 图完成处理和 aclgraph Capture。

`npugraph_ex` 提供模型编译缓存方案，可通过 [`cache_compile`](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph%5Fex/api/inference/cache%5Fcompile.md) 将首次编译结果持久化。后续运行命中缓存后可跳过 Dynamo 和 Guards 阶段，从而降低图模式的启动开销。

![npugraph_ex执行时间分布示意图](./images/npugraph_ex_execution_time_distribution_v2.png)

以 LLaMA 2-70B 为例，上图呈现了启动与未开启模型编译缓存的耗时分布：

- **原始推理任务执行**，分为 5 个阶段：
  - **Dynamo**：Python 级 JIT 编译器，重写 Python 字节码，将 PyTorch 操作序列提取到 FX 图中。
  - **Guards**：Dynamo 编译生成 Guards，在每次执行前执行，用于区分程序是否需要被重新捕获与编译。
  - **aclgraph Capture**：npugraph_ex 捕获 Stream 任务到 Device 侧。
  - **Input 处理**：更新图内 input 类参数的输入地址；如果输入 Tensor 使用私有格式（如 `FRACTAL_NZ`），其格式信息会被保留。
  - **Replay**：Device 基于给定的输入进行真正的计算并得到输出结果。
- **开启模型编译缓存**：通过缓存 Dynamo 这个耗时占比最大环节，实现模型的加速启动。

## 2. 使用约束

- 本功能支持的产品型号参见[使用说明](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/overview.md#%E4%BD%BF%E7%94%A8%E8%AF%B4%E6%98%8E)。
- 如果图中包含依赖随机数生成器（RNG）的算子（例如 `randn`、`bernoulli`、`dropout` 等），不支持使用本功能。
- 本功能跳过了 Dynamo 的 JIT 编译环节和 Guards 环节，与 `torch.compile` 原始方案相比多了如下限制：
  - 缓存要与执行计算图一一对应，若重编译则缓存失效。
  - Guards 阶段被跳过且不会触发 JIT 编译，要求生成模型的脚本和加载缓存的脚本一致。
  - CANN 包跨版本缓存无法保证兼容性；版本升级后需要清理缓存目录，并重新运行脚本生成缓存。
- `func` 必须是 Module 实例的方法（即 `method`），且该方法未被其他装饰器修饰。
- `func` 必须能形成整图，即必须支持 `fullgraph=True`。
- `func` 只能被触发一次 Dynamo trace，如果 `func` 发生重编译则会放弃缓存。
- 使用 `cache_compile` 接口后，原先脚本中的 `torch.compile` 编译流程不再需要。

## 3. 使用方法

### 3.1 准备 PyTorch 模型脚本

原始模型脚本使用 `torch.compile` 进行编译，如下所示（简化示例）：

```python
import torch
import torch_npu

class Model(torch.nn.Module):
    def forward(self, x, kv):
        return self.linear(x.data) + self.linear(kv[0])

model = Model().npu()
compiled_model = torch.compile(model, backend="npugraph_ex", dynamic=False, fullgraph=True)
res = compiled_model(x, kv)
```

### 3.2 改造模型脚本

由于 `forward` 函数会根据不同场景（如 prompt / decode）触发多次重编译，需要为每个场景封装独立的 `func` 函数，然后通过 `cache_compile` 接口实现编译缓存。

改造步骤：
1. 将 `forward` 函数的实现提取为 `_forward` 函数。
2. 为每个场景（如 prompt、decode）封装新的 `func` 函数，直接调用 `_forward`。
3. 在 `__init__` 中通过 `cache_compile` 接口为每个 `func` 创建缓存编译版本。
4. 在 `forward` 中添加调用逻辑，根据场景分发到对应的缓存函数。
5. 移除 `torch.compile` 调用，直接执行 `model`。

### 3.3 运行并生成缓存

首次运行时会生成各 `func` 函数的缓存文件。缓存文件路径由 `cache_compile` 中 `cache_dir` 参数指定：

- 若 `cache_dir` 为**绝对路径**，缓存文件路径为 `{cache_dir}/{model_info}/{func}`。
- 若 `cache_dir` 为**相对路径**，缓存文件路径为 `{work_dir}/{cache_dir}/{model_info}/{func}`。

`cache_dir` 默认为 `.torchair_cache`（若无会新建，请确保有读写权限）。

> 若编译缓存的模型涉及多机多卡，缓存路径还包含集合通信相关的 `world_size` 和 `global_rank` 信息，路径为 `{work_dir}/{cache_dir}/{model_info}/world{world_size}global_rank{global_rank}/{func}`。

### 3.4 再次执行验证

再次执行脚本时，系统会从缓存目录加载编译结果并跳过 Dynamo 和 Guards 阶段。必须在脚本、计算图、CANN 版本和并行拓扑保持一致的前提下比较首次生成缓存与再次命中缓存的端到端耗时，并结合日志或缓存文件确认实际命中。

## 4. 使用示例

下面的示例展示如何改造模型脚本以使用编译缓存功能。`InputMeta` 为仿照 vLLM 框架的入参结构，模型根据 `is_prompt` 字段区分 prompt 和 decode 两个场景，分别使用独立的缓存编译函数。

> **首次运行**会生成缓存文件到 `.torchair_cache` 目录；**再次运行**时会命中缓存，跳过编译阶段。

In [ ]:
import dataclasses
from typing import List
import torch
import torch_npu


# InputMeta 为仿照 vLLM 框架的入参结构
@dataclasses.dataclass
class InputMeta:
    data: torch.Tensor
    is_prompt: bool


class CachedModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(8, 8)
        # 通过 cache_compile 为每个场景创建缓存编译版本
        # prompt 和 decode 分别封装，避免不同场景共用同一缓存入口。
        self.cached_prompt = torch.npu.npugraph_ex.inference.cache_compile(
            self.prompt, cache_dir=".torchair_cache"
        )
        self.cached_decode = torch.npu.npugraph_ex.inference.cache_compile(
            self.decode, cache_dir=".torchair_cache"
        )

    # 原始计算逻辑提取到 _forward
    def _forward(self, x: InputMeta, kv: List[torch.Tensor]):
        return self.linear(x.data) + self.linear(kv[0])

    # 为 prompt 场景封装独立函数
    def prompt(self, x, kv):
        return self._forward(x, kv)

    # 为 decode 场景封装独立函数
    def decode(self, x, kv):
        return self._forward(x, kv)

    # forward 中根据场景分发到对应的缓存函数
    def forward(self, x, kv):
        if x.is_prompt:
            return self.cached_prompt(x, kv)
        return self.cached_decode(x, kv)


# 模型使用 float16 并迁移到 NPU，与下方半精度输入保持一致。
model = CachedModel().half().npu()

# 执行 prompt 场景（首次运行会生成缓存）
x = InputMeta(torch.randn(2, 8, dtype=torch.float16).npu(), True)
kv = [torch.randn(2, 8, dtype=torch.float16).npu()]
prompt_out = model(x, kv)  # 首次调用生成 prompt 场景缓存

# 执行 decode 场景（首次运行会生成缓存）
x.is_prompt = False
decode_out = model(x, kv)  # 首次调用生成 decode 场景缓存

print("prompt output:", tuple(prompt_out.shape))
print("decode output:", tuple(decode_out.shape))
print("\n缓存已生成到 .torchair_cache 目录，再次运行将命中缓存。")

## 5. 查看缓存文件（可选）

如需查看封装的 `func` 函数缓存文件 `compiled_module`，可通过 [readable_cache](https://gitcode.com/Ascend/torchair/blob/master/docs/zh/npugraph%5Fex/api/inference/readable%5Fcache.md) 接口读取。`compiled_module` 主要存储了 `torch.compile` 成图过程中模型脚本、模型结构、执行流程等相关信息，可用于问题定位分析。

```python
torch.npu.npugraph_ex.inference.readable_cache(
    "/path/to/.torchair_cache/.../prompt/compiled_module",
    file="prompt.py"
)
```

> `compiled_module` 内容最终解析到可读文件 `prompt.py`（格式不限，如 `.py`、`.txt` 等）中。

## 6. 缓存目录结构说明

缓存目录结构如下：
```
.torchair_cache/                          # 默认缓存目录（cache_dir）
└── {model_info}/                         # 模型信息（自动包含 aclgraphcache 关键词）
    ├── prompt/                           # prompt 场景的缓存
    │   └── compiled_module               # 编译产物
    └── decode/                           # decode 场景的缓存
        └── compiled_module               # 编译产物
```

> **多机多卡场景**：缓存路径还会包含 `world{world_size}global_rank{global_rank}` 层级，不要在不同并行拓扑之间盲目复用缓存。

> **版本升级**：CANN 跨版本缓存无法保证兼容性，升级后需要清理缓存目录并重新生成。

## 7. 课后练习

### 一、单选题

（1）【单选题】`cache_compile` 命中编译缓存后，主要跳过 `torch.compile` 流程中的哪些阶段？
- A. Dynamo 和 Guards
- B. Replay 和 Input 处理
- C. 模型前向计算和输出生成
- D. NPU Kernel 执行

（2）【单选题】使用 `cache_compile` 时，`func` 必须满足哪项要求？
- A. 可以是任意被装饰器修饰的普通函数
- B. 必须是 Module 实例的方法，且未被其他装饰器修饰
- C. 只能是 `__init__` 方法
- D. 必须在 CPU 上执行

（3）【单选题】`cache_compile` 对计算图的要求是？
- A. 只能使用 `fullgraph=False`
- B. 必须支持 `fullgraph=True`，形成整图
- C. 必须包含随机数算子
- D. 必须每次触发图中断

（4）【单选题】当 `cache_dir` 使用相对路径时，缓存目录相对于哪里解析？
- A. Python 安装目录
- B. 当前工作目录 `work_dir`
- C. NPU 驱动目录
- D. 模型权重目录

（5）【单选题】下列哪种情况通常需要清理并重新生成模型编译缓存？
- A. CANN 版本升级
- B. 仅修改日志输出级别
- C. 增加一次预热次数
- D. 查看缓存目录

（6）【单选题】如果一个 `func` 在运行中发生重编译，`cache_compile` 的行为是？
- A. 自动复用旧缓存继续运行
- B. 放弃该函数的缓存
- C. 自动改用多流表达
- D. 删除全部模型参数

### 二、多选题

（7）【多选题】下列哪些情况会使模型编译缓存不适用、失效或不兼容？
- A. 图中包含依赖 RNG 的算子
- B. 生成缓存与加载缓存的脚本不一致
- C. CANN 跨版本复用缓存
- D. `func` 无法形成整图

（8）【多选题】验证编译缓存是否生效时，合理的做法有哪些？
- A. 分别记录首次生成缓存与再次命中缓存的端到端耗时
- B. 查看日志或缓存目录，确认发生缓存命中
- C. 保持脚本、CANN 版本和运行环境一致
- D. 只测量首次执行时间即可得出缓存收益

（9）【多选题】关于缓存目录结构和路径，正确的说法有哪些？
- A. 默认缓存目录为 `.torchair_cache`
- B. 缓存路径会区分模型信息和具体的 func
- C. 多机多卡缓存路径还会包含 `world_size` 与 `global_rank` 信息
- D. 所有不同拓扑的缓存都可以直接共用

（10）【多选题】将普通 `torch.compile` 脚本改造为缓存编译脚本时，通常包括哪些步骤？
- A. 将原有计算逻辑提取为 `_forward`
- B. 为 prompt、decode 等场景封装独立的 func
- C. 在 `__init__` 中对各 func 调用 `cache_compile`
- D. 移除原有的 `torch.compile` 编译调用

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/04.02_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
